# 04 — Full-data check: do the 50k results hold on all 1.09M rows?
The same pipeline as notebooks 01–03, but on **all 1,088,589 normal transactions** instead of the 50k sample: same cleaning, same 5% injection (seed 42), same features, same models and split.
If the scores match the 50k results, the sample gives the same conclusions, which is the evidence for using it.

**Run order and time (on a recent laptop):** sections 1–4 take about 10–20 minutes. Section 5 (LOF) and section 6 (One-Class SVM) are **optional and slow**: run them last, One-Class SVM ideally overnight. Needs about 4–6 GB of free memory, so close other apps.

In [1]:
import sys, time, gc
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))     # project root, so `config` and `src` can be imported
import config
import numpy as np
import pandas as pd
from src import prepare as prep
from src.inject import make_dirty
from src.features import reference, build_features, feature_sets
from src.models import run_all

## 1. Clean the full dataset (same steps as notebook 01, without sampling)

In [2]:
t = time.time()
normal = prep.remove_laundering(prep.load_raw())
full, fields = prep.clean_table(normal)          # metadata parsing, field screening, column removal
del normal; gc.collect()                         # free memory
print('Full clean table:', full.shape, f'({time.time() - t:.0f}s)')
print('Metadata fields kept:', (fields['decision'] == 'keep').sum(), 'of', len(fields))
prep.quality_checks(full)

Full clean table: (1088589, 22) (58s)
Metadata fields kept: 11 of 22


{'missing_values': 0,
 'duplicate_rows': 0,
 'balance_errors': 0,
 'negative_or_zero_amounts': 0,
 'negative_balances': 0,
 'hour_matches_timestamp': 1.0,
 'cities_in_more_than_one_state': 0}

The same 11 metadata fields are kept and every quality check passes on all 1.09M rows, so the full data is as clean as the sample.

## 2. Inject the same 5% of errors (seed 42) and build the features

In [3]:
t = time.time()
ref = reference(full)
dirty = make_dirty(full, rate=0.05, seed=42)
X, y, issue = build_features(dirty, ref)
X_sets = feature_sets(X)
rate = y.mean()
del full, dirty; gc.collect()
print(f'Rows: {len(y):,} | anomalies: {int(y.sum()):,} ({rate:.2%}) | features: {X.shape[1]} ({time.time() - t:.0f}s)')
print(issue.value_counts())

Rows: 1,097,661 | anomalies: 54,432 (4.96%) | features: 63 (21s)
issue_type
none           1043229
crossfield        9072
numerical         9072
missing           9072
balance           9072
categorical       9072
duplicate         9072
Name: count, dtype: int64


## 3. The scalable models on the full data
Isolation Forest, Random Forest and XGBoost, with and without the check features, using a stratified 70/30 split (about 768,000 training and 329,000 test rows).

In [4]:
t = time.time()
full_results, full_per_issue = run_all(X_sets, y, issue, rate, only=['Isolation Forest', 'Random Forest', 'XGBoost'])
print(f'Done in {(time.time() - t) / 60:.1f} min')
full_results.to_csv(config.RESULTS / 'full_data_results.csv', index=False)
full_results.round(3)

Done in 1.9 min


,features,model,type,precision,recall,F1,PR-AUC,seconds
0,with checks,Isolation Forest,unsupervised,0.160,0.159,0.160,0.117,4.128
1,with checks,Random Forest,supervised,1.000,1.000,1.000,1.000,45.101
2,with checks,XGBoost,supervised,1.000,1.000,1.000,1.000,2.667
3,raw only,Isolation Forest,unsupervised,0.050,0.050,0.050,0.048,4.433
4,raw only,Random Forest,supervised,0.748,0.346,0.473,0.397,52.259
5,raw only,XGBoost,supervised,0.793,0.582,0.671,0.617,3.400


## 4. Compare with the 50k sample (from notebook 03)

In [5]:
sample = pd.read_csv(config.RESULTS / 'baseline_results.csv')
key = ['features', 'model']
compare = (sample[key + ['PR-AUC', 'recall', 'precision']]
           .merge(full_results[key + ['PR-AUC', 'recall', 'precision', 'seconds']], on=key, suffixes=(' 50k', ' full')))
compare['PR-AUC difference'] = compare['PR-AUC full'] - compare['PR-AUC 50k']
compare.round(3)

,features,model,PR-AUC 50k,recall 50k,precision 50k,PR-AUC full,recall full,precision full,seconds,PR-AUC difference
0,with checks,Isolation Forest,0.158,0.238,0.231,0.117,0.159,0.160,4.128,-0.041
1,with checks,Random Forest,1.000,1.000,1.000,1.000,1.000,1.000,45.101,0.000
2,with checks,XGBoost,0.999,0.999,1.000,1.000,1.000,1.000,2.667,0.001
3,raw only,Isolation Forest,0.049,0.048,0.046,0.048,0.050,0.050,4.433,-0.001
4,raw only,Random Forest,0.230,0.101,0.439,0.397,0.346,0.748,52.259,0.167
5,raw only,XGBoost,0.469,0.473,0.542,0.617,0.582,0.793,3.400,0.148


**How to read it:** a small PR-AUC difference (for example within ±0.05) means the 50k sample gives the same result as the full data.

In the test run, the **with-checks** results matched (Random Forest and XGBoost 1.00 on both), and Isolation Forest was close. The **raw-only supervised** models did **better** on the full data (Random Forest about 0.23 → 0.40, XGBoost higher too): with 20 times more rows they see far more examples of the rare error patterns. That is a finding, not a failure. Report it: the 50k sample keeps the conclusions (checks dominate, same ranking of models) but **underestimates** how well supervised models learn raw-only patterns from more data.

In [6]:
full_per_issue.round(2)

issue_type,balance,categorical,crossfield,duplicate,missing,none,numerical
Isolation Forest | with checks,0.12,0.04,0.15,0.22,0.04,0.04,0.37
Random Forest | with checks,1.00,1.00,1.00,1.00,1.00,0.00,1.00
XGBoost | with checks,1.00,1.00,1.00,1.00,1.00,0.00,1.00
Isolation Forest | raw only,0.05,0.02,0.05,0.06,0.03,0.05,0.10
Random Forest | raw only,0.00,0.45,0.35,0.00,0.32,0.01,0.96
XGBoost | raw only,0.01,1.00,0.50,0.01,0.99,0.01,0.99


## 5. Optional: LOF on the full data (slow)
LOF must compare every test row with its nearest neighbours among about 768,000 training rows. Expect roughly 10–60 minutes depending on the computer. Raw-only features, where LOF did best on the sample.

In [7]:
t = time.time()
lof_results, _ = run_all({'raw only': X_sets['raw only']}, y, issue, rate, only=['LOF'])
lof_results['wall_minutes'] = (time.time() - t) / 60
lof_results.to_csv(config.RESULTS / 'full_data_lof.csv', index=False)
lof_results.round(3)

,features,model,type,precision,recall,F1,PR-AUC,seconds,wall_minutes
0,raw only,LOF,unsupervised,0.285,0.291,0.288,0.259,701.477,11.699


## 6. Optional: One-Class SVM on the full data (run overnight)
The timing test estimated about an hour to **fit** One-Class SVM on the full data (fit time grows with the square of the rows), plus extra time to score the test rows. Run this last, leave the laptop plugged in, and stop it with **Kernel → Interrupt** if needed. Whatever happens (finishes, or runs for hours), the time is real evidence for the sample size.

In [8]:
t = time.time()
print('Started at', time.strftime('%H:%M'))
svm_results, _ = run_all({'with checks': X_sets['with checks']}, y, issue, rate, only=['One-Class SVM'])
svm_results['wall_minutes'] = (time.time() - t) / 60
svm_results.to_csv(config.RESULTS / 'full_data_ocsvm.csv', index=False)
print('Finished at', time.strftime('%H:%M'))
svm_results.round(3)

Started at 19:24
Finished at 21:02


,features,model,type,precision,recall,F1,PR-AUC,seconds,wall_minutes
0,with checks,One-Class SVM,unsupervised,0.694,0.694,0.694,0.645,5897.927,98.312


## Notes for the paper
- **Same cleaning, same injection, same split rule:** only the number of rows differs, so any difference in scores comes from sample size.
- **Report:** the comparison table (section 4), and the wall-clock times of every model, including LOF and One-Class SVM, or the time after which One-Class SVM was stopped.
- **Justification sentence (adapt to your numbers):** *"On the full 1.09M rows, Isolation Forest, Random Forest and XGBoost reached PR-AUC within ±0.0X of their 50k results, while One-Class SVM took X hours for a single fit. The 50k sample therefore preserves the conclusions while making the repeated runs (3 severities × 3 seeds, tuning, SHAP) feasible for all five models."*